# TP 3: AMARI MODEL / EQUILIBRIUM CASE Ib: - W_m < h < - W_infty
## Contact: emre.baspinar@inria.fr

In [ ]:
#####################################################################################
## TP 3: AMARI MODEL / CASE Ib: - W_m < h < - W_infty                     ################
#####################################################################################

# Contact: emre.baspinar@inria.fr

####################################################################################
####################################################################################


#####################################################################
## Initialization #################################################
#####################################################################


import numpy as np
from scipy.integrate import quad, cumulative_trapezoid
import matplotlib.pyplot as plt
from scipy.signal import convolve

#---------------------------------------------------------------------#
#---------------------------------------------------------------------#


## Simulation parameters

In [ ]:
######################################################################
## Simulation setup ##################################################
######################################################################

# Simulation parameters
L = 100.0                          # Length of the cortical layer 
N = 2**13+1                        # Number of discretization nodes
dx = L / (N-1)                     # Space between discretization nodes
x = np.linspace(-L/2, L/2, N)      # Spatial vector of discretized cortical layer

mu = 1                             # Time scale
dt = 0.1                           # Time step
T  = 10000.0                       # Final time
n_steps = int(T / dt)              # number of time steps

## Population transfer functions

In [ ]:
## Transfer function (Heaviside step function)
def s(u):
    return (u > 0).astype(float)

## Plot the transfer function
u = np.linspace(-5, 5, 20)  # vector of u values for the transfer function
sVec = s(u)
plt.figure(figsize=(7,4))
plt.plot(u, sVec, label="Transfer function")
plt.axhline(0, color='k', linestyle="--")
plt.xlabel("u")
plt.ylabel("s(x)")
plt.title("Population transfer function")
plt.legend()
plt.tight_layout()
plt.show()

## Connnectivity kernel

In [ ]:
## Connectivity kernel

## Analytic Mexican Hat
# scaleVal = 3.0
# def kernel_mexican_hat(x, sigma=scaleVal): # Classical Mexican Hat formula
#     return (2 / (np.sqrt(3*sigma) * np.pi**0.25)) * (1 - (x**2 / sigma**2)) * np.exp(-x**2 / (2*sigma**2))

# x_shifted = (x - L/2) % L  # periodic shift: needed since the kernel is not centered around x=0.
# w = kernel_mexican_hat(x_shifted)

# Parametric Mexican Hat
sigma_excVal = 2        # range of excitation
sigma_inhVal = 4        # range of inhibition
beta_excVal = 2.0       # strength of excitation
beta_inhVal = 0.7       # strength of inhibition

def kernel_mexican_hat(x, sigma_exc = sigma_excVal, sigma_inh = sigma_inhVal, beta_exc = beta_excVal, beta_inh = beta_inhVal): # Parameteric Mexican Hat as a difference of Gaussian functions (DoG)
    return beta_exc * np.exp(-x**2 / (2 * sigma_exc**2)) - beta_inh * np.exp(-x**2 / (2 * sigma_inh**2))

w = kernel_mexican_hat(x)

## Plot the connectivity kernel
plt.figure(figsize=(7,4))
plt.plot(x, w, label="Mexican-hat kernel")
plt.axhline(0, color='k', linestyle="--")
plt.xlabel("x")
plt.ylabel("w(x)")
plt.title("Mexican-hat kernel in 1D")
plt.legend()
plt.tight_layout()
plt.show()

## Integral W(x) of the connectivity kernel

In [ ]:
######################################################################
## Compute the integral W(x) of the connectivity kernel ##############
######################################################################

## Compute W(x) for all x values using cumulative trapezoid
# x values of the right half plane
x_vals_right = np.linspace(0, int(L/2), int(N/2)+1)

# Cumulative integral from 0
W_cumulative = cumulative_trapezoid(w[int(N/2):N], x_vals_right, initial = 0)  # initial=0 sets W(0)=0

# Find W_m and W_infty, and print their values
W_m = np.max(W_cumulative) # max value of W(x)
W_infty = W_cumulative[-1] # value of W(\infty)
print("W_m = ", W_m)
print("2 W_m = ", 2 * W_m)
print("W(∞) = ", W_infty)
print("2 W(∞) = ", 2 * W_infty)

## Plot W(x)
plt.figure(figsize=(7,4))
plt.plot(x_vals_right, W_cumulative, label="W(x) = ∫₀ˣ w(y) dy")
plt.xlim(0, L/2)
# plt.ylim(-1,1)
plt.axhline(0, color='k', linestyle='--')

# Highlight W_m
plt.axhline(W_m, color='red', linestyle='--', label=f"W_m = {W_m:.3f}")
plt.text(L/2*0.6, W_m, f"W_m = {W_m:.3f}", color='red', va='bottom')

# Highlight W_infty
plt.axhline(W_infty, color='blue', linestyle='--', label=f"W(∞) = {W_infty:.3f}")
plt.text(L/2*0.6, W_infty, f"W(∞) = {W_infty:.3f}", color='blue', va='bottom')

plt.xlabel("x")
plt.ylabel("Value")
plt.title("Cumulative integral W(x)")
plt.legend()
plt.tight_layout()
plt.show()

## Localized initial stimulation

In [ ]:
## Define step function which we will use to define the bump (localized stimulation)
def step_function(x, x0):
    """Heaviside step function: 0 for x < x0, 1 for x >= x0"""
    return np.where(x < x0, 0, 1)

## Define the function modeling a localized stimulation
def bump_function(x, a, b, amp = 1.0):
    """
    Creates a localized bump on interval [a, b]
    using amplitude amp and sum of two step functions: amp * (H(x-a) - H(x-b))
    """
    return amp * (step_function(x, a) - step_function(x, b))

## Plot an example of localized initial condition (or initial stimulation in terms of Amari's terminology)
# Define bump parameters for plotted example
x1 = -1.0  # bump starts at x = x1
x2 = 1.0  # bump ends at x = x2

# Generate the initial localized stimulation (this will give the initial condition to the model)
bump = bump_function(x, x1, x2)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(x, bump, linewidth=2, label=f'Bump function on [{x1}, {x2}]')
# plt.axvline(x=x1, color='r', linestyle='--', alpha=0.7, label=f'x = {x1}')
# plt.axvline(x=x2, color='g', linestyle='--', alpha=0.7, label=f'x = {x2}')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Localized Bump Function using Step Functions')
plt.legend()
# plt.grid(True, alpha=0.3)
plt.grid(False)
plt.ylim(-0.1, 1.1)
plt.show()

# Print some information
print(f"Bump function is 1 on the interval [{x1}, {x2}]")
print(f"Bump function is 0 elsewhere")

## Simulations
## a1-solution
## Step 1
a) In which part of the solution taxonomy table (Fig 9 in the lecture notes) are the a1-solutions? Use the threshold stimulus value "h" to identify it.

b) Decrease the initial localized stimulation width "width". What do you observe? Which solution type appears as you decrease "width"? Why?

c) Increase the initial localized stimulation width "width". What do you observe? Which solution type appears as you incrase "width"? Why?

d) What is these two cases mean in terms of stability of a1-solutions?

In [ ]:
##############################################################################################
## CASE Ib:   a1-solution (unstable),           -W_m < h < -W_infty        #################
##############################################################################################
## W_m = 2.445,     2 * W_m = 4.890,        W_infty = 1.504,      2*W_infty = 3.008 ##########
##############################################################################################

## Threshold stimulus
h = -1.9952845 # h for unstable a1-solution (unstable)
print(f" h = {h}")
print()

## Find the length "x = a" from cumulative integral W(x) = h for the chosen h value 
tolIndex = 1*1e-3           # error tolerance to detect W(a)
index_length_bump = np.where((W_cumulative > -h - tolIndex) & (W_cumulative < -h + tolIndex))[0] # index of W(a)
length_bump = index_length_bump * dx # length a
print("index corresponding to W(x) = h:", index_length_bump)
print()
print("a:", length_bump)
print()
print("W(a) =", W_cumulative[index_length_bump])
print()

## Initial condition
width = length_bump[0] # = length "a"
# width = length_bump[0] - 0.02 # due to unstability: explosion towards φ-solution
# width = length_bump[0] + 0.02 # explosion towards ∞-solution does not occur since W_infty + h > 0 is not satisfied (see (11) in [Amari, Biological Cybernetics 1977])
center = 0
amplitudeBump = 1
u0 = bump_function(x, center - width/2, center + width/2, amp = amplitudeBump)

## External stimulus
P_ext = np.zeros_like(x) # Since we consider an equilibrium solution


## Simulation routine
def simulate(u0, h, n_steps, dt, mu, tol=1e-6, check_every=50):
    u = u0.copy()
    last_u = u.copy()
    # eps = 1e-6

    for tstep in range(n_steps):
        transferFunctionOutput = s(u)
        conv = convolve(transferFunctionOutput, w, mode='same') * dx
        du = (-u + conv + h + P_ext) / mu
        u = u + dt * du

        # Check the convergence: if diff < tol, then the system has converged to an equilibrium.
        if (tstep + 1) % check_every == 0:
            diff = np.linalg.norm(u - last_u) / (np.linalg.norm(u) + 1e-12)
            if diff < tol:
                break
            last_u = u.copy()

    # Categories of the equilibrium solutions
    if np.max(u) <= 0:
        eq = "φ"
    elif np.min(u) > 0:
        eq = "∞"
    else:
        eq = "a"
    return u, eq

#######################################################################
## Simulate and Plot ##################################################
#######################################################################

## Initialize the plot
results = []
plt.figure(figsize=(8, 5))

## Simulate and save the solution at T to u_final
u_final, eq = simulate(u0, h, n_steps, dt, mu)      
results.append((h, eq, u_final))

## Show the length a
# Find indexes where elements are greater than 0
positiveElements = np.where(u_final > 0)[0]

# Length of a
length_positiveElements = len(positiveElements) * dx
print("Length a:", length_positiveElements)

## Plot the final result
plt.plot(x, u_final, label=f"h={h:.3f} → {eq}")
plt.xlabel("x")
plt.ylabel("u(x)")
plt.ylim(-10, 10)
plt.title(f"Equilibrium profile for {eq}-solution")

# Highlight y=0
plt.axhline(0, color="k", linestyle="--", linewidth=1, label="u=0")

plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# Print classification table
print("Equilibrium solution type:")
for h, eq, _ in results:
    print(f"  h={h:.3f} → {eq}")

## a2-solution
## Step 2
a) In which part of the solution taxonomy table (Fig 9 in the lecture notes) are the a2-solutions? Use the threshold stimulus value "h" to identify it.

b) Decrease the initial localized stimulation width "width". What do you observe? Which solution type appears as you decrease "width"? Why?

c) Increase the initial localized stimulation width "width". What do you observe? Which solution type appears as you incrase "width"? Why?

d) What is these two cases mean in terms of stability of a2-solutions?

e) What does stability change in the a1- and a2-solutions regarding the small perturbations introduced in the length of initial localized stimulation?

In [ ]:
##############################################################################################
## CASE Ib:   a2-solution (stable),            -W_m < h < -W_infty         #################
##############################################################################################
## W_m = 2.445,     2 * W_m = 4.890,        W_infty = 1.504,      2*W_infty = 3.008 ##########
##############################################################################################

## Threshold stimulus
h = -2.00432252 # h for a2-solution (stable)
print(f" h = {h}")
print()

## Find the length "x = a" from cumulative integral W(x) = h for the chosen h value 
tolIndex = 1*1e-3           # error tolerance to detect W(a)
index_length_bump = np.where((W_cumulative > -h - tolIndex) & (W_cumulative < -h + tolIndex))[0] # index of W(a)
length_bump = index_length_bump * dx # length a
print("index corresponding to W(x) = h:", index_length_bump)
print()
print("a:", length_bump)
print()
print("W(a) =", W_cumulative[index_length_bump])
print()

## Initial condition
width = length_bump[1] # = length "a"
# width = length_bump[1] + 2 # due to stability: the same a2-solution 
# width = length_bump[1] - 2 # due to stability: the same a2-solution 
center = 0
amplitudeBump = 1
u0 = bump_function(x, center - width/2, center + width/2, amp = amplitudeBump)
# u0 = 100* np.ones_like(x)

## External stimulus
P_ext = np.zeros_like(x) # Since we consider an equilibrium solution


## Simulation routine
def simulate(u0, h, n_steps, dt, mu, tol=1e-6, check_every=50):
    u = u0.copy()
    last_u = u.copy()
    # eps = 1e-6

    for tstep in range(n_steps):
        transferFunctionOutput = s(u)
        conv = convolve(transferFunctionOutput, w, mode='same') * dx
        du = (-u + conv + h + P_ext) / mu
        u = u + dt * du

        # Check the convergence: if diff < tol, then the system has converged to an equilibrium.
        if (tstep + 1) % check_every == 0:
            diff = np.linalg.norm(u - last_u) / (np.linalg.norm(u) + 1e-12)
            if diff < tol:
                break
            last_u = u.copy()

    # Categories of the equilibrium solutions
    if np.max(u) <= 0:
        eq = "φ"
    elif np.min(u) > 0:
        eq = "∞"
    else:
        eq = "a"
    return u, eq

#######################################################################
## Simulate and Plot ##################################################
#######################################################################

## Initialize the plot
results = []
plt.figure(figsize=(8, 5))

## Simulate and save the solution at T to u_final
u_final, eq = simulate(u0, h, n_steps, dt, mu)      
results.append((h, eq, u_final))

## Show the length a
# Find indexes where elements are greater than 0
positiveElements = np.where(u_final > 0)[0]

# Length of a
length_positiveElements = len(positiveElements) * dx
print("Length a:", length_positiveElements)

## Plot the final result
plt.plot(x, u_final, label=f"h={h:.3f} → {eq}")
plt.xlabel("x")
plt.ylabel("u(x)")
plt.ylim(-10, 10)
plt.title(f"Equilibrium profile for {eq}-solution")

# Highlight y=0
plt.axhline(0, color="k", linestyle="--", linewidth=1, label="u=0")

plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# Print classification table
print("Equilibrium solution type:")
for h, eq, _ in results:
    print(f"  h={h:.3f} → {eq}")

##  φ-solution
## Step 3
In which part of the solution taxonomy table (Fig 9 in the lecture notes) are the  φ-solutions? Use the threshold stimulus value "h" to identify it.

In [ ]:
##############################################################################################
## CASE Ib:            φ-solution,             -W_m < h < -W_infty         #################
##############################################################################################
## W_m = 2.445,     2 * W_m = 4.890,        W_infty = 1.504,      2*W_infty = 3.008 ##########
##############################################################################################

## Threshold stimulus
h = -1.9952845 # h for φ-solution
print(f" h = {h}")
print()

## Find the length "x = a" from cumulative integral W(x) = h for the chosen h value 
tolIndex = 1*1e-2           # error tolerance to detect W(a)
index_length_bump = np.where((W_cumulative > -h - tolIndex) & (W_cumulative < -h + tolIndex))[0] # index of W(a)
length_bump = index_length_bump * dx # length a
print("index corresponding to W(x) = h:", index_length_bump)
print()
print("a:", length_bump)
print()
print("W(a) =", W_cumulative[index_length_bump])
print()

## Initial condition
# width = length_bump[0] # = length "a"
width = length_bump[0] - 0.1 # due to unstability: explosion towards φ-solution
center = 0
amplitudeBump = 1
u0 = bump_function(x, center - width/2, center + width/2, amp = amplitudeBump)
# u0 = 100* np.ones_like(x)

## External stimulus
P_ext = np.zeros_like(x) # Since we consider an equilibrium solution


## Simulation routine
def simulate(u0, h, n_steps, dt, mu, tol=1e-6, check_every=50):
    u = u0.copy()
    last_u = u.copy()
    # eps = 1e-6

    for tstep in range(n_steps):
        transferFunctionOutput = s(u)
        conv = convolve(transferFunctionOutput, w, mode='same') * dx
        du = (-u + conv + h + P_ext) / mu
        u = u + dt * du

        # Check the convergence: if diff < tol, then the system has converged to an equilibrium.
        if (tstep + 1) % check_every == 0:
            diff = np.linalg.norm(u - last_u) / (np.linalg.norm(u) + 1e-12)
            if diff < tol:
                break
            last_u = u.copy()

    # Categories of the equilibrium solutions
    if np.max(u) <= 0:
        eq = "φ"
    elif np.min(u) > 0:
        eq = "∞"
    else:
        eq = "a"
    return u, eq

#######################################################################
## Simulate and Plot ##################################################
#######################################################################

## Initialize the plot
results = []
plt.figure(figsize=(8, 5))

## Simulate and save the solution at T to u_final
u_final, eq = simulate(u0, h, n_steps, dt, mu)      
results.append((h, eq, u_final))

## Show the length a
# Find indexes where elements are greater than 0
positiveElements = np.where(u_final > 0)[0]

# Length of a
length_positiveElements = len(positiveElements) * dx
print("Length a:", length_positiveElements)

## Plot the final result
plt.plot(x, u_final, label=f"h={h:.3f} → {eq}")
plt.xlabel("x")
plt.ylabel("u(x)")
plt.ylim(-10, 10)
plt.title(f"Equilibrium profile for {eq}-solution")

# Highlight y=0
plt.axhline(0, color="k", linestyle="--", linewidth=1, label="u=0")

plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# Print classification table
print("Equilibrium solution type:")
for h, eq, _ in results:
    print(f"  h={h:.3f} → {eq}")

#---------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------#

#########################################################################################
######################################## THE END ########################################
#########################################################################################